# Retrieval-Augmented Image Captioning – Evaluation

Generates captions for the **test set** using batched beam search with Accelerate multi-process support,
then saves predictions in a format ready for `pycocoevalcap`.

## 0. Environment (Kaggle fallback)

In [ ]:
# !git clone https://github.com/vuthetam/RAG_Captioning.git

# import os
# import sys
# sys.path.append("/kaggle/working/RAG_Captioning")

In [ ]:
# %%writefile RAG_Captioning/main.py

# import os
# from pathlib import Path

# _defaults = {
#     "DATASET_COCO_PATH" : "/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json",
#     "IMAGES_PATH"        : "/kaggle/input/datasets/vuthetam/mscoco-2014/images",
#     "LOAD_BEST_CHECKPOINT_DIR": "/kaggle/input/notebooks/vuthetam/rag-captioning/checkpoints",
#     "DMODEL"         : "512",
#     "NHEADS"         : "8",
#     "NLAYERS"        : "4",
#     "BATCH_SIZE"     : "32",
#     "MAX_LENGTH"     : "40",
#     "BEAM_SIZE"      : "5",
#     "NUM_WORKERS"    : "4",
#     "DROPOUT"        : "0.1",
# }
# os.environ.update(_defaults)

## 1. Imports & Accelerator

In [ ]:
# %%writefile -a RAG_Captioning/main.py

import json
import sys
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from accelerate import Accelerator
from accelerate.utils import set_seed

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import (
    TEST_DF_PATH,
    VOCAB_PATH,
    DMODEL,
    NHEADS,
    NLAYERS,
    BATCH_SIZE,
    MAX_LENGTH,
    BEAM_SIZE,
    NUM_WORKERS,
    DROPOUT,
    LOAD_BEST_CHECKPOINT_PATH,
    ARTIFACTS_DIR,
    PREDICTIONS_PATH,
)
from src.vocabulary import Vocabulary
from src.dataset import ImageOnlyDataset, create_clip_transform
from src.encoder import CLIPViTB16Encoder
from src.decoder import TransformerCaptionDecoder
from src.checkpoint import load_checkpoint
from src.inference import generate_captions

accelerator = Accelerator(mixed_precision="fp16")
set_seed(42)

accelerator.print(f"Device : {accelerator.device}")

## 2. Load Data & Vocabulary

In [ ]:
# %%writefile -a RAG_Captioning/main.py

test_df = pd.read_parquet(TEST_DF_PATH)
vocab   = Vocabulary.load(VOCAB_PATH)

accelerator.print(f"test_df : {len(test_df):,} images")
accelerator.print(f"Vocabulary size : {len(vocab):,}")

## 3. Dataset & DataLoader

In [ ]:
# %%writefile -a RAG_Captioning/main.py

test_dataset = ImageOnlyDataset(test_df, transform=create_clip_transform())

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # preserve order for matching with test_df rows
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

accelerator.print(f"test batches : {len(test_loader):,}")

## 4. Model & Checkpoint

In [ ]:
# %%writefile -a RAG_Captioning/main.py

encoder = CLIPViTB16Encoder(d_model=DMODEL)
decoder = TransformerCaptionDecoder(
    vocab_size=len(vocab),
    d_model=DMODEL,
    nhead=NHEADS,
    num_layers=NLAYERS,
    dropout=DROPOUT,
    max_length=MAX_LENGTH,
    pad_idx=vocab.pad_idx(),
)

if LOAD_BEST_CHECKPOINT_PATH and LOAD_BEST_CHECKPOINT_PATH.exists():
    load_checkpoint(
        LOAD_BEST_CHECKPOINT_PATH,
        encoder,
        decoder,
        device=accelerator.device,
    )
    accelerator.print(f"Loaded checkpoint: {LOAD_BEST_CHECKPOINT_PATH}")
else:
    accelerator.print("WARNING: no checkpoint found, using random weights")

## 5. Accelerator prepare

In [ ]:
# %%writefile -a RAG_Captioning/main.py

encoder, decoder, test_loader = accelerator.prepare(encoder, decoder, test_loader)

## 6. Generate Captions

In [ ]:
# %%writefile -a RAG_Captioning/main.py

captions = generate_captions(
    encoder=encoder,
    decoder=decoder,
    dataloader=test_loader,
    vocab=vocab,
    beam_size=BEAM_SIZE,
    max_length=MAX_LENGTH,
    accelerator=accelerator,
    show_progress=True,
)

if accelerator.is_main_process:
    accelerator.print(f"Generated {len(captions):,} captions")

## 7. Save Results

In [ ]:
# %%writefile -a RAG_Captioning/main.py

if accelerator.is_main_process:
    # Build prediction list: [{"image_id": imgid, "caption": "..."}]
    # captions is a list of token lists — join into strings
    imgids = test_df["imgid"].tolist()

    predictions = [
        {"image_id": int(imgid), "caption": " ".join(tokens)}
        for imgid, tokens in zip(imgids, captions)
    ]

    with open(PREDICTIONS_PATH, "w") as f:
        json.dump(predictions, f, indent=2)

    accelerator.print(f"Saved {len(predictions):,} predictions → {PREDICTIONS_PATH}")
    accelerator.print("\nSample predictions:")
    for p in predictions[:5]:
        accelerator.print(f"  [{p['image_id']}] {p['caption']}")

In [ ]:
# !accelerate launch RAG_Captioning/main.py